# Fourier basis — how the ray directions depend on the seed

Supports Supplementary Note 1 (*An even set of rays*, *Why not a Sobol' draw
of directions*) and Fig. S1. For a given number of rays $R$, how much does the
coverage of directions change from one seed to the next?

A direction only matters as an axis ($\pm\hat{\mathbf u}$ give the same mode),
so every direction is taken to the lower hemisphere. Two measures, in degrees:

* **largest gap**: the largest angle from any axis to its nearest ray. It
  describes the set as a whole: the worst-served orientation.
* **angle to a fixed axis**: the angle from one arbitrary axis, the same for
  every seed, to its nearest ray. It describes what one orientation in the
  data gets.

Three ways to draw $R$ directions, each the frozen `omega_spatial` buffer of a
real `FSF` (or plain random draws):

* **Sobol'**: scrambled Sobol' coordinates through the inverse normal;
* **Fibonacci**: `stratify_directions=True` (what `n_directions` uses), a
  hemispherical Fibonacci lattice turned by a seeded random rotation;
* **independent**: $R$ independent uniform axes.

Writes `results/direction_seeds.npz`, read by `s01d_fourier_basis_figures`.
Runs in `curlew_dev`.

In [1]:
from pathlib import Path

import numpy as np
import torch

import sys
sys.path.insert(0, str(Path("..").resolve()))  # publication/: figstyle, diagnostics
import diagnostics

import curlew
from curlew.core import HSet
from curlew.fields.series import FSF

curlew.device = "cpu"
curlew.dtype = torch.float64

RES_DIR = Path("./results")
RES_DIR.mkdir(parents=True, exist_ok=True)

## Directions and measures

The wavelength band only sets $\lVert\boldsymbol\omega\rVert$; the directions
are what is tested. The largest gap is measured on an equal-area grid of test
axes on the lower hemisphere, much finer than any gap here (its spacing is
printed).

In [2]:
BAND = (2000.0, 5000.0)
METHODS = ("Sobol'", "Fibonacci", "independent")


def directions(method, seed, n):
    # n unit wavevector directions of one 3-D bank.
    if method == "independent":
        a = np.random.default_rng(seed).normal(size=(n, 3))
        return a / np.linalg.norm(a, axis=1, keepdims=True)
    field = FSF(
        "f", H=HSet(), input_dim=3, rff_features=n, length_scale_range=BAND,
        seed=seed, stratify_directions=(method == "Fibonacci"),
    )
    om = field.omega_spatial.detach().cpu().numpy()
    return om / np.linalg.norm(om, axis=1, keepdims=True)


NET_N = 501
xx, yy, rr, nn = diagnostics._schmidt_grid(NET_N)
TEST = nn[rr <= 1.0]
print(f"{len(TEST)} test axes, spacing ~{np.degrees(np.sqrt(2 * np.pi / len(TEST))):.2f} deg")

TARGET = np.array([0.31, -0.52, -0.80])   # an arbitrary fixed axis
TARGET = TARGET / np.linalg.norm(TARGET)


def nearest_angle(points, axes):
    # Angle (deg) from each point to its nearest axis, +u and -u identified.
    best = np.abs(points @ axes.T).max(axis=1)
    return np.degrees(np.arccos(np.clip(best, -1.0, 1.0)))


def largest_gap(axes):
    return nearest_angle(TEST, axes).max()


def target_angle(axes):
    return float(nearest_angle(TARGET[None, :], axes)[0])

196317 test axes, spacing ~0.32 deg


## Spread across seeds

The numbers of rays of the 3-D fits (Table S1), 200 seeds each.

In [3]:
R_VALUES = (16, 32, 64)
N_SEEDS = 200

gap = {(m, R): np.array([largest_gap(directions(m, s, R)) for s in range(N_SEEDS)])
       for m in METHODS for R in R_VALUES}
target = {(m, R): np.array([target_angle(directions(m, s, R)) for s in range(N_SEEDS)])
          for m in METHODS for R in R_VALUES}


def describe(x):
    p5, p50, p95 = np.percentile(x, [5, 50, 95])
    return f"{p50:5.1f} [{p5:5.1f}, {p95:5.1f}]  max {x.max():5.1f}  spread {np.ptp(x):5.2f}"


for name, data in (("Largest gap (deg)", gap), ("Angle to the fixed axis (deg)", target)):
    print(f"{name}: median [5th, 95th percentile], maximum and spread over {N_SEEDS} seeds")
    for R in R_VALUES:
        for m in METHODS:
            print(f"  R={R:>2} {m:>12}: {describe(data[(m, R)])}")
    print()

Largest gap (deg): median [5th, 95th percentile], maximum and spread over 200 seeds
  R=16       Sobol':  37.6 [ 32.9,  44.3]  max  47.0  spread 16.24
  R=16    Fibonacci:  27.8 [ 27.8,  27.9]  max  27.9  spread  0.18
  R=16  independent:  43.0 [ 36.1,  52.9]  max  64.9  spread 33.17
  R=32       Sobol':  28.6 [ 25.4,  34.6]  max  39.9  spread 16.80
  R=32    Fibonacci:  21.2 [ 21.2,  21.3]  max  21.3  spread  0.20
  R=32  independent:  32.9 [ 27.8,  41.1]  max  44.9  spread 19.87
  R=64       Sobol':  21.4 [ 18.8,  24.5]  max  27.1  spread  9.01
  R=64    Fibonacci:  14.3 [ 14.3,  14.4]  max  14.4  spread  0.31
  R=64  independent:  25.3 [ 21.9,  31.2]  max  34.1  spread 13.78

Angle to the fixed axis (deg): median [5th, 95th percentile], maximum and spread over 200 seeds
  R=16       Sobol':  14.6 [  5.8,  28.5]  max  37.9  spread 35.95
  R=16    Fibonacci:  14.1 [  4.8,  22.5]  max  25.4  spread 24.66
  R=16  independent:  16.8 [  4.6,  33.0]  max  46.2  spread 44.20
  R=32       So

## One set of rays for the nets

One seed, $R = 32$, for panels a and b of Fig. S1.

In [4]:
R_NET = 32
NET_SEED = 404
net = {m: directions(m, NET_SEED, R_NET) for m in ("Sobol'", "Fibonacci")}
for m, a in net.items():
    print(f"{m}, R={R_NET}, seed {NET_SEED}: largest gap {largest_gap(a):.1f} deg")

Sobol', R=32, seed 404: largest gap 28.7 deg
Fibonacci, R=32, seed 404: largest gap 21.1 deg


## Save

In [5]:
key = {"Sobol'": "sobol", "Fibonacci": "fibonacci", "independent": "independent"}
arrays = {"r_values": np.array(R_VALUES), "target": TARGET,
          "net_r": R_NET, "net_seed": NET_SEED,
          "net_sobol": net["Sobol'"], "net_fibonacci": net["Fibonacci"]}
for m in METHODS:
    for R in R_VALUES:
        arrays[f"gap_{key[m]}_{R}"] = gap[(m, R)]
        arrays[f"target_{key[m]}_{R}"] = target[(m, R)]
np.savez(RES_DIR / "direction_seeds.npz", **arrays)
print("wrote", RES_DIR / "direction_seeds.npz")

wrote results/direction_seeds.npz
